# IndiGo DCF — InterGlobe Aviation (NSE: INDIGO)
## With War Scenario Stress Tests

**WACC:** 11% | **Terminal Growth Rate:** 4.5% | **Projection Period:** 5 years  
Data sourced live from Yahoo Finance via `yfinance`

In [ ]:
# --------------------------------------------------------------------------
# INVESTMENT SUMMARY — run this cell AFTER executing all cells below
# --------------------------------------------------------------------------
from IPython.display import HTML, display
import datetime

TODAY = datetime.date.today().strftime('%d %b %Y')

if upside_pct >= 15:
    call, badge_color = 'BULLISH', '#27AE60'
elif upside_pct <= -10:
    call, badge_color = 'BEARISH', '#C0392B'
else:
    call, badge_color = 'NEUTRAL', '#E8900A'

upside_label = f"{upside_pct:+.1f}% {'upside' if upside_pct >= 0 else 'downside'}"

display(HTML(f"""
<div style="background:#F0F4FF; border-left:6px solid #1B1464; padding:18px 24px;
            border-radius:6px; margin:12px 0; font-family: sans-serif;">
  <h3 style="margin:0 0 6px 0; color:#1B1464;">
    Investment Summary — InterGlobe Aviation (NSE: INDIGO)
  </h3>
  <p style="margin:0 0 10px 0; color:#666; font-size:0.9em;">
    DCF-based analysis &nbsp;|&nbsp; WACC: {WACC*100:.0f}% &nbsp;|&nbsp;
    TGR: {TGR*100:.1f}% &nbsp;|&nbsp; As of {TODAY}
  </p>
  <p style="margin:0 0 12px 0;">
    <span style="background:{badge_color}; color:white; font-weight:bold;
                 padding:3px 10px; border-radius:4px; font-size:1.0em;">
      {call}
    </span>
    &nbsp;&nbsp;
    Implied Price: <strong>&#8377;{implied_price:,.0f}</strong>
    &nbsp;|&nbsp; Market Price: <strong>&#8377;{current_price:,.0f}</strong>
    &nbsp;|&nbsp; <strong>{upside_label}</strong>
  </p>
  <p style="margin:0; line-height:1.6; color:#222;">
    Our 5-year DCF on InterGlobe Aviation (IndiGo) generates an implied share price of
    <strong>&#8377;{implied_price:,.0f}</strong>, representing <strong>{upside_label}</strong>
    vs the current market price of &#8377;{current_price:,.0f}. IndiGo commands ~60%
    domestic market share and operates a cost-efficient single-type A320neo fleet,
    structurally positioned to benefit from India's secular aviation demand growth
    (domestic PAX CAGR ~12% projected). The base case assumes revenue growth tapering
    from 12% to 8% over 5 years with gradual EBITDA margin expansion as ancillary revenue
    and fleet density improve. Key risks are binary: a crude oil spike to $120/bbl compresses
    implied value to <strong>&#8377;{s2['Implied Price (&#8377;)']:,.0f}</strong>
    ({s2['vs Base Case %']:+.1f}% vs base); a severe demand shock (load factor 72%)
    drives it to <strong>&#8377;{s3['Implied Price (&#8377;)']:,.0f}</strong>
    ({s3['vs Base Case %']:+.1f}% vs base). Given IndiGo's dominant market position,
    improving unit economics, and India's underpenetrated aviation market, the risk-reward
    tilts {'favourably at current valuations' if upside_pct > 5 else
           'unfavourably given limited margin of safety' if upside_pct < -5 else
           'roughly in balance, warranting a hold'}.
  </p>
  <p style="margin:8px 0 0 0; color:#888; font-size:0.82em; font-style:italic;">
    Source: Yahoo Finance via yfinance &nbsp;|&nbsp; Not investment advice
  </p>
</div>
"""))

---
## Step 1 — Pull Live Financial Data

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', palette='muted')

TICKER = 'INDIGO.NS'
ticker = yf.Ticker(TICKER)

# --- Raw financials ---
income_stmt = ticker.financials          # annual income statement
balance_sheet = ticker.balance_sheet     # annual balance sheet
cash_flow = ticker.cashflow              # annual cash flow statement
info = ticker.info

print(f"Fetched data for: {info.get('longName', TICKER)}")
print(f"Currency: {info.get('currency', 'INR')}")
print(f"Exchange: {info.get('exchange', 'NSI')}")

In [ ]:
# --------------------------------------------------------------------------
# Helper: safely extract a row from a DataFrame, return NaN series if missing
# --------------------------------------------------------------------------
def safe_row(df, candidates):
    """Try each label in candidates; return the first match or NaN series."""
    for label in candidates:
        if label in df.index:
            return df.loc[label]
    return pd.Series(np.nan, index=df.columns if df is not None and not df.empty else [])


# --------------------------------------------------------------------------
# Extract key line items (last 5 fiscal years, newest → oldest)
# --------------------------------------------------------------------------
cols = income_stmt.columns[:5]  # up to 5 most-recent years

revenue      = safe_row(income_stmt,   ['Total Revenue', 'Revenue'])[cols]
ebit         = safe_row(income_stmt,   ['EBIT', 'Operating Income', 'Operating Income Loss'])[cols]
da           = safe_row(cash_flow,     ['Depreciation And Amortization',
                                        'Depreciation Amortization Depletion',
                                        'Reconciled Depreciation'])[cols]
ebitda       = ebit + da

capex        = safe_row(cash_flow,     ['Capital Expenditure',
                                        'Purchase Of PPE',
                                        'Capital Expenditures'])[cols].abs()

# Working capital change (use operating cash flow proxy if unavailable)
wc_change    = safe_row(cash_flow,     ['Changes In Working Capital',
                                        'Change In Working Capital'])[cols]

tax_rate     = 0.25   # effective corporate tax rate (India)
nopat        = ebit * (1 - tax_rate)
fcf          = nopat + da - capex - wc_change

# --------------------------------------------------------------------------
# Market data
# --------------------------------------------------------------------------
current_price      = info.get('currentPrice') or info.get('regularMarketPrice', np.nan)
shares_outstanding = info.get('sharesOutstanding', np.nan)
market_cap         = info.get('marketCap', np.nan)

total_debt         = safe_row(balance_sheet, ['Total Debt',
                                               'Long Term Debt',
                                               'Total Long Term Debt']).iloc[0]
cash               = safe_row(balance_sheet, ['Cash And Cash Equivalents',
                                               'Cash Cash Equivalents And Short Term Investments',
                                               'Cash And Short Term Investments']).iloc[0]
net_debt           = total_debt - cash

# --------------------------------------------------------------------------
# Summary table  (values in INR Crores)
# --------------------------------------------------------------------------
CR = 1e7   # 1 Crore = 10 million

summary = pd.DataFrame({
    'Revenue (₹ Cr)':      (revenue / CR).round(0),
    'EBITDA (₹ Cr)':       (ebitda  / CR).round(0),
    'EBIT (₹ Cr)':         (ebit    / CR).round(0),
    'D&A (₹ Cr)':          (da      / CR).round(0),
    'CapEx (₹ Cr)':        (capex   / CR).round(0),
    'ΔWorking Capital (₹ Cr)': (wc_change / CR).round(0),
    'NOPAT (₹ Cr)':        (nopat   / CR).round(0),
    'FCF (₹ Cr)':          (fcf     / CR).round(0),
}).T

summary.columns = [str(c.year) for c in summary.columns]

print("="*65)
print(f"  InterGlobe Aviation (INDIGO.NS) — Historical Financials")
print("="*65)
print(summary.to_string())
print()
print(f"  Current Price      : ₹{current_price:,.2f}")
print(f"  Shares Outstanding : {shares_outstanding/1e7:,.2f} Cr shares")
print(f"  Market Cap         : ₹{market_cap/CR:,.0f} Cr")
print(f"  Total Debt         : ₹{total_debt/CR:,.0f} Cr")
print(f"  Cash               : ₹{cash/CR:,.0f} Cr")
print(f"  Net Debt           : ₹{net_debt/CR:,.0f} Cr")
print("="*65)

---
## Step 2 — Base Case DCF Model

In [ ]:
# --------------------------------------------------------------------------
# DCF assumptions — Base Case
# --------------------------------------------------------------------------
WACC         = 0.11   # 11% — INR-denominated, Indian aviation risk
TGR          = 0.045  # 4.5% terminal growth rate
PROJ_YEARS   = 5

# Use most-recent year as base
base_revenue = float(revenue.iloc[0])
base_ebitda  = float(ebitda.iloc[0])
base_ebitda_margin = base_ebitda / base_revenue if base_revenue else 0.18

# Analyst-style growth & margin assumptions (Base Case)
revenue_growth   = [0.12, 0.11, 0.10, 0.09, 0.08]   # tapering growth
ebitda_margins   = [base_ebitda_margin + 0.01 * i for i in range(PROJ_YEARS)]  # gradual expansion
da_pct_rev       = float(da.iloc[0]) / base_revenue if base_revenue else 0.10
capex_pct_rev    = float(capex.iloc[0]) / base_revenue if base_revenue else 0.08
wc_pct_rev       = 0.01   # modest working capital build

# --------------------------------------------------------------------------
# Project FCFs
# --------------------------------------------------------------------------
years       = list(range(1, PROJ_YEARS + 1))
proj_rev    = []
proj_ebitda = []
proj_ebit   = []
proj_fcf    = []

rev = base_revenue
for i in range(PROJ_YEARS):
    rev    = rev * (1 + revenue_growth[i])
    _ebitda = rev * ebitda_margins[i]
    _da     = rev * da_pct_rev
    _ebit   = _ebitda - _da
    _nopat  = _ebit * (1 - tax_rate)
    _capex  = rev * capex_pct_rev
    _wc     = rev * wc_pct_rev
    _fcf    = _nopat + _da - _capex - _wc
    proj_rev.append(rev)
    proj_ebitda.append(_ebitda)
    proj_ebit.append(_ebit)
    proj_fcf.append(_fcf)

# --------------------------------------------------------------------------
# Terminal value & enterprise value
# --------------------------------------------------------------------------
terminal_fcf = proj_fcf[-1] * (1 + TGR)
terminal_val = terminal_fcf / (WACC - TGR)

pv_fcf = sum(fcf_t / (1 + WACC)**t for t, fcf_t in zip(years, proj_fcf))
pv_tv  = terminal_val / (1 + WACC)**PROJ_YEARS
ev     = pv_fcf + pv_tv

equity_val      = ev - net_debt
implied_price   = equity_val / shares_outstanding if shares_outstanding else np.nan
upside_pct      = (implied_price / current_price - 1) * 100 if current_price else np.nan

# --------------------------------------------------------------------------
# Print DCF summary
# --------------------------------------------------------------------------
proj_df = pd.DataFrame({
    'Revenue (₹ Cr)':      [r/CR for r in proj_rev],
    'EBITDA (₹ Cr)':       [e/CR for e in proj_ebitda],
    'EBITDA Margin %':     [e/r*100 for e,r in zip(proj_ebitda, proj_rev)],
    'FCF (₹ Cr)':          [f/CR for f in proj_fcf],
}, index=[f'FY+{y}' for y in years])

print("="*65)
print("  Base Case — Projected Financials")
print("="*65)
print(proj_df.round(1).to_string())
print()
print(f"  PV of FCFs          : ₹{pv_fcf/CR:>10,.0f} Cr")
print(f"  PV of Terminal Value: ₹{pv_tv/CR:>10,.0f} Cr")
print(f"  Enterprise Value    : ₹{ev/CR:>10,.0f} Cr")
print(f"  Less: Net Debt      : ₹{net_debt/CR:>10,.0f} Cr")
print(f"  Equity Value        : ₹{equity_val/CR:>10,.0f} Cr")
print()
print(f"  Implied Share Price : ₹{implied_price:>10,.2f}")
print(f"  Current Price       : ₹{current_price:>10,.2f}")
print(f"  Upside / (Downside) : {upside_pct:>+10.1f}%")
print("="*65)

---
## Step 3 — War Scenario Stress Tests

In [ ]:
# --------------------------------------------------------------------------
# WACC × Terminal Growth Rate Sensitivity — Implied Share Price (₹)
# Uses base-case FCF projections; varies discount rate and terminal growth
# --------------------------------------------------------------------------
from IPython.display import display

wacc_range = [0.09, 0.10, 0.11, 0.12, 0.13]
tgr_range  = [0.030, 0.035, 0.040, 0.045, 0.050, 0.055, 0.060]

sens_data = {}
for w in wacc_range:
    row = {}
    for g in tgr_range:
        col_lbl = f'{g*100:.1f}%'
        if w <= g:
            row[col_lbl] = float('nan')
            continue
        tv_s    = proj_fcf[-1] * (1 + g) / (w - g)
        pv_f_s  = sum(f / (1 + w)**t for t, f in enumerate(proj_fcf, 1))
        pv_tv_s = tv_s / (1 + w)**PROJ_YEARS
        ev_s    = pv_f_s + pv_tv_s
        eq_s    = ev_s - net_debt
        p       = eq_s / shares_outstanding if shares_outstanding else float('nan')
        row[col_lbl] = round(p, 0)
    sens_data[f'{w*100:.0f}%'] = row

sens_df = pd.DataFrame(sens_data).T
sens_df.index.name = 'WACC \\ TGR →'

# Identify base case cell for highlight
base_row_lbl = f'{int(WACC * 100)}%'
base_col_lbl = f'{TGR * 100:.1f}%'

def _highlight_base(df):
    s = pd.DataFrame('', index=df.index, columns=df.columns)
    if base_row_lbl in df.index and base_col_lbl in df.columns:
        s.loc[base_row_lbl, base_col_lbl] = (
            'background-color: #1B1464; color: white; '
            'font-weight: bold; font-size: 12pt; '
            'border: 3px solid #F5A623 !important;'
        )
    return s

print(f"\n  WACC Sensitivity Table — Implied Share Price (₹)")
print(f"  Base Case cell (navy): WACC={base_row_lbl}, TGR={base_col_lbl}")
print(f"  Current Market Price : ₹{current_price:,.0f}\n")

display(
    sens_df.style
        .background_gradient(cmap='RdYlGn', axis=None)
        .apply(_highlight_base, axis=None)
        .format(lambda x: f'₹{x:,.0f}' if pd.notna(x) and not (isinstance(x, float) and x != x) else '—')
        .set_caption(
            f'Implied Share Price (₹) · Base Case highlighted in navy '
            f'· Current Price ₹{current_price:,.0f}'
        )
        .set_table_styles([
            {'selector': 'caption',
             'props': [('color', '#1B1464'), ('font-weight', 'bold'),
                       ('font-size', '11pt'), ('margin-bottom', '8px')]},
            {'selector': 'th.col_heading',
             'props': [('background-color', '#1B1464'), ('color', 'white'),
                       ('font-weight', 'bold'), ('text-align', 'center'),
                       ('padding', '6px 14px')]},
            {'selector': 'th.row_heading',
             'props': [('background-color', '#1B1464'), ('color', 'white'),
                       ('font-weight', 'bold'), ('padding', '6px 14px')]},
            {'selector': 'td',
             'props': [('text-align', 'center'), ('padding', '6px 14px'),
                       ('font-size', '10.5pt'), ('border', '1px solid #E0E0E0')]},
        ])
)

In [ ]:
# --------------------------------------------------------------------------
# Chart 1 — Scenario Implied Prices vs Current Price
# IndiGo brand palette: navy #1B1464, blue #2855AE, amber #E8900A, red #C0392B
# --------------------------------------------------------------------------
import datetime

NAVY    = '#1B1464'
BLUE    = '#2855AE'
AMBER   = '#E8900A'
CRIMSON = '#C0392B'
BG      = '#F7F9FC'
TODAY   = datetime.date.today().strftime('%d %b %Y')
DATA_NOTE = f'Source: Yahoo Finance via yfinance  |  {TODAY}'

scenarios        = [s1, s2, s3]
names            = [s['Scenario'] for s in scenarios]
prices           = [s['Implied Price (₹)'] for s in scenarios]
vs_market_pct    = [s['vs Current Price %'] for s in scenarios]
scenario_colors  = [BLUE, AMBER, CRIMSON]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG)
fig.suptitle('IndiGo DCF — War Scenario Stress Test', fontsize=14,
             fontweight='bold', color=NAVY, y=1.01)

# --- Left: Implied prices ---
ax1 = axes[0]
ax1.set_facecolor(BG)
bars = ax1.barh(names, prices, color=scenario_colors, edgecolor='white', height=0.5)
ax1.axvline(current_price, color=NAVY, linestyle='--', linewidth=1.8,
            label=f'Current Price: ₹{current_price:,.0f}')
ax1.set_xlabel('Implied Share Price (₹)', fontsize=11, labelpad=8, color=NAVY)
ax1.set_title('Implied Share Price by Scenario', fontsize=12, fontweight='bold',
              color=NAVY, pad=10)
ax1.legend(fontsize=10)
for bar, price in zip(bars, prices):
    ax1.text(bar.get_width() + max(prices) * 0.01, bar.get_y() + bar.get_height() / 2,
             f'₹{price:,.0f}', va='center', fontsize=10, fontweight='bold', color=NAVY)
ax1.set_xlim(0, max(prices) * 1.22)
ax1.invert_yaxis()
ax1.tick_params(labelsize=9)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

# --- Right: % vs current market price ---
ax2 = axes[1]
ax2.set_facecolor(BG)
bar_colors2 = [BLUE if v >= 0 else CRIMSON for v in vs_market_pct]
bars2 = ax2.barh(names, vs_market_pct, color=bar_colors2, edgecolor='white', height=0.5)
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('% vs Current Market Price', fontsize=11, labelpad=8, color=NAVY)
ax2.set_title('Upside / Downside vs Current Price (%)', fontsize=12,
              fontweight='bold', color=NAVY, pad=10)
for bar, val in zip(bars2, vs_market_pct):
    offset = 0.5 if val >= 0 else -0.5
    ax2.text(val + offset, bar.get_y() + bar.get_height() / 2,
             f'{val:+.1f}%', va='center',
             ha='left' if val >= 0 else 'right',
             fontsize=10, fontweight='bold', color=NAVY)
ax2.invert_yaxis()
ax2.tick_params(labelsize=9)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.0f}%'))

fig.text(0.5, -0.02, DATA_NOTE, ha='center', fontsize=8, color='gray', style='italic')
plt.tight_layout()
plt.savefig('scenario_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: scenario_comparison.png")

# --------------------------------------------------------------------------
# Chart 2 — Projected Free Cash Flow by Scenario over 5 years
# --------------------------------------------------------------------------
import datetime

NAVY    = '#1B1464'
BLUE    = '#2855AE'
AMBER   = '#E8900A'
CRIMSON = '#C0392B'
BG      = '#F7F9FC'
TODAY   = datetime.date.today().strftime('%d %b %Y')
DATA_NOTE = f'Source: Yahoo Finance via yfinance  |  {TODAY}'

def get_fcf_series(revenue_growth_rates, ebitda_margin_adj, capex_pct=None, wc_pct=0.01):
    _capex_pct = capex_pct if capex_pct is not None else capex_pct_rev
    rev = base_revenue
    fcfs = []
    for i in range(PROJ_YEARS):
        rev     = rev * (1 + revenue_growth_rates[i])
        _ebitda = rev * ebitda_margin_adj[i]
        _da     = rev * da_pct_rev
        _ebit   = _ebitda - _da
        _nopat  = _ebit * (1 - tax_rate)
        _capex  = rev * _capex_pct
        _wc     = rev * wc_pct
        fcfs.append((_nopat + _da - _capex - _wc) / CR)
    return fcfs

fcf_base   = get_fcf_series([0.12, 0.11, 0.10, 0.09, 0.08],
                             [base_ebitda_margin + 0.01 * i for i in range(5)])
fcf_oil    = get_fcf_series(oil_shock_growth, oil_shock_margin)
fcf_demand = get_fcf_series(demand_growth, demand_margin)

yr_labels = [f'FY+{i}' for i in range(1, 6)]

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.plot(yr_labels, fcf_base,   'o-',  color=BLUE,    linewidth=2.5, markersize=9,
        label='Base Case', markerfacecolor='white', markeredgewidth=2.2)
ax.plot(yr_labels, fcf_oil,    's--', color=AMBER,   linewidth=2.5, markersize=9,
        label='Oil Shock ($120/bbl)', markerfacecolor='white', markeredgewidth=2.2)
ax.plot(yr_labels, fcf_demand, '^:',  color=CRIMSON, linewidth=2.5, markersize=9,
        label='Demand Collapse (LF 72%)', markerfacecolor='white', markeredgewidth=2.2)

# End-of-period annotations
for series, color in [(fcf_base, BLUE), (fcf_oil, AMBER), (fcf_demand, CRIMSON)]:
    ax.annotate(f'₹{series[-1]:,.0f} Cr',
                xy=(4, series[-1]), xytext=(4.1, series[-1]),
                fontsize=9, color=color, fontweight='bold', va='center')

ax.axhline(0, color='#333333', linewidth=0.9, linestyle='-')
ax.set_title('Projected Free Cash Flow by Scenario', fontsize=14,
             fontweight='bold', color=NAVY, pad=12)
ax.set_xlabel('Projection Year', fontsize=11, labelpad=8, color=NAVY)
ax.set_ylabel('Free Cash Flow (₹ Crores)', fontsize=11, labelpad=8, color=NAVY)
ax.legend(fontsize=10, framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f} Cr'))
ax.tick_params(axis='both', labelsize=10)
ax.grid(True, alpha=0.35, color='#BBBBBB', linestyle='--')

fig.text(0.5, -0.02, DATA_NOTE, ha='center', fontsize=8, color='gray', style='italic')
plt.tight_layout()
plt.savefig('fcf_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fcf_scenarios.png")

In [ ]:
# --------------------------------------------------------------------------
# Chart 3 — Value Bridge: Base Case → Stress Scenario Implied Prices
# --------------------------------------------------------------------------
import datetime

NAVY    = '#1B1464'
BLUE    = '#2855AE'
AMBER   = '#E8900A'
CRIMSON = '#C0392B'
GREEN   = '#27AE60'
BG      = '#F7F9FC'
TODAY   = datetime.date.today().strftime('%d %b %Y')
DATA_NOTE = f'Source: Yahoo Finance via yfinance  |  {TODAY}'

base_price = s1['Implied Price (₹)']
oil_delta  = s2['Implied Price (₹)'] - base_price
dem_delta  = s3['Implied Price (₹)'] - base_price

labels = [
    'Base Case',
    'Oil Shock\nImpact',
    'Oil Shock\nPrice',
    'Demand Collapse\nImpact',
    'Demand Collapse\nPrice',
]
values  = [base_price, oil_delta, s2['Implied Price (₹)'], dem_delta, s3['Implied Price (₹)']]
bottoms = [0,          base_price, 0,                       base_price, 0]
bar_colors = [
    BLUE,
    GREEN if oil_delta >= 0 else CRIMSON,
    AMBER,
    GREEN if dem_delta >= 0 else CRIMSON,
    CRIMSON,
]

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

all_abs = [abs(v) for v in values]
y_pad = max(all_abs) * 0.07

for i, (lbl, val, bot, clr) in enumerate(zip(labels, values, bottoms, bar_colors)):
    is_delta = 'Impact' in lbl
    height   = abs(val) if is_delta else val
    bottom   = bot if is_delta else 0
    ax.bar(i, height, bottom=bottom, color=clr, edgecolor='white', width=0.55, alpha=0.9)
    label_y = (bot + val + y_pad) if is_delta else (val + y_pad)
    ax.text(i, label_y,
            f'₹{val:+,.0f}' if is_delta else f'₹{val:,.0f}',
            ha='center', fontsize=9.5, fontweight='bold', color=NAVY)

ax.axhline(current_price, color=NAVY, linestyle='--', linewidth=1.8,
           label=f'Current Market Price: ₹{current_price:,.0f}')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9.5)
ax.set_ylabel('Implied Share Price (₹)', fontsize=11, labelpad=8, color=NAVY)
ax.set_title('Value Bridge — Base Case vs War Stress Scenarios',
             fontsize=14, fontweight='bold', color=NAVY, pad=12)
ax.legend(fontsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.grid(True, axis='y', alpha=0.3, color='#BBBBBB', linestyle='--')
ax.set_ylim(bottom=min(0, min(values)) - y_pad * 2)

fig.text(0.5, -0.03, DATA_NOTE, ha='center', fontsize=8, color='gray', style='italic')
plt.tight_layout()
plt.savefig('value_bridge.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: value_bridge.png")

In [ ]:
# --------------------------------------------------------------------------
# Summary — Scenario Results with Actual Prices
# --------------------------------------------------------------------------
from IPython.display import Markdown, display
import datetime

TODAY = datetime.date.today().strftime('%d %b %Y')

rows = "\n".join(
    f"| {s['Scenario']} | ₹{s['Implied Price (₹)']:,.0f} "
    f"| {s['vs Base Case %']:+.1f}% "
    f"| {s['vs Current Price %']:+.1f}% |"
    for s in [s1, s2, s3]
)

display(Markdown(f"""
---
## Summary

*As of {TODAY} &nbsp;|&nbsp; WACC: {WACC*100:.0f}% &nbsp;|&nbsp;
TGR: {TGR*100:.1f}% &nbsp;|&nbsp;
Current Market Price: ₹{current_price:,.0f}*

| Scenario | Implied Price | vs Base Case | vs Market Price |
|---|---|---|---|
{rows}

**Key Takeaways:**
- IndiGo's high fuel cost exposure (~35–40% of opex) makes it acutely sensitive to oil price shocks.
- A demand collapse scenario (load factor 72%, revenue −22%) represents the most severe downside
  case given IndiGo's fixed cost structure and high operating leverage.
- The base case DCF assumes gradual margin improvement as fleet utilisation and ancillary revenue mature.
- All scenarios should be read as directional stress indicators, not investment advice.
"""))

In [ ]:
# --------------------------------------------------------------------------
# Chart 3 — Waterfall: Value bridge from Base Case to each stress scenario
# --------------------------------------------------------------------------
base_price = s1['Implied Price (₹)']
oil_delta  = s2['Implied Price (₹)'] - base_price
dem_delta  = s3['Implied Price (₹)'] - base_price

labels  = ['Base Case', 'Oil Shock Impact', 'Oil Shock Price',
           'Demand Collapse Impact', 'Demand Collapse Price']
values  = [base_price, oil_delta, s2['Implied Price (₹)'],
           dem_delta,  s3['Implied Price (₹)']]
bottoms = [0,          base_price, 0,
           base_price, 0]
bar_c   = ['#2ecc71', '#e74c3c', '#e67e22',
           '#e74c3c', '#c0392b']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (lbl, val, bot, clr) in enumerate(zip(labels, values, bottoms, bar_c)):
    ax.bar(i, abs(val) if lbl.endswith('Impact') else val,
           bottom=bot if lbl.endswith('Impact') else 0,
           color=clr, edgecolor='white', width=0.55)
    y_pos = bot + val/2 if lbl.endswith('Impact') else val/2
    ax.text(i, max(values)*1.03 if not lbl.endswith('Impact') else bot + val + max(values)*0.01,
            f'₹{val:+,.0f}' if lbl.endswith('Impact') else f'₹{val:,.0f}',
            ha='center', fontsize=9, fontweight='bold')

ax.axhline(current_price, color='navy', linestyle='--', linewidth=1.5,
           label=f'Current Price ₹{current_price:,.0f}')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Share Price (₹)', fontsize=11)
ax.set_title('Value Bridge — Base Case vs War Stress Scenarios', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('value_bridge.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: value_bridge.png")

---
## Summary

| Scenario | Implied Price | vs Base | vs Market |
|---|---|---|---|
| Base Case | See above | — | See above |
| Oil Shock ($120 crude) | See above | See above | See above |
| Demand Collapse (LF 72%) | See above | See above | See above |

**Key Takeaways:**
- IndiGo's high fuel cost exposure (~35-40% of opex) makes it acutely sensitive to oil price shocks.
- A demand collapse scenario (load factor 72%, revenue -22%) represents the most severe downside case given IndiGo's fixed cost structure and high operating leverage.
- The base case DCF assumes gradual margin improvement as fleet utilisation and ancillary revenue mature.
- All scenarios should be read as directional stress indicators, not investment advice.